# nbskill behavior tests

This notebook is an executable contract for the public tools documented in the other notebooks. It builds temporary notebooks, edits them, runs them, reviews them, and converts a small Python file without touching the repository notebooks.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import _in_call_parse

from nbskill.convert import convert
from nbskill.execute import exec_nb
from nbskill.foundation import demo_path, remove_demo_path
from nbskill.mcp import capture_call, create_mcp
from nbskill.read import context
from nbskill.review import diff_nb
from nbskill.write import split_nb_chapter, update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    return start


project_root = _find_project_root()

## Build a notebook fixture

Every test below works against a temporary notebook. This mirrors the way agents should experiment: create a small fixture, prove the tool behavior, then apply the same tool to the real notebook.

In [ ]:
root = demo_path("test_nbskill_contract")
root.mkdir()
demo = root / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("Notebook note.", cell_type="markdown"),
    mk_cell("import math", cell_type="code"),
    mk_cell(chr(10).join(["## Setup", "A markdown section for context."]), cell_type="markdown"),
    mk_cell(chr(10).join(["value = 3", "value"]), cell_type="code"),
    mk_cell(chr(10).join([
        "#| export",
        "def demo_fn(x):",
        "    \"\"\"Return the input.\"\"\"",
        "    return x",
        "",
        "class DemoBox:",
        "    \"\"\"Box a value.\"\"\"",
        "    def get(self):",
        "        \"\"\"Return the boxed value.\"\"\"",
        "        return 3",
    ]), cell_type="code"),
]), demo)
assert demo.exists()

## Reading and symbol documentation

`context` should expose useful notebook structure without raw JSON and should also find a symbol inside one of this repository's source notebooks.

In [ ]:
file_text = capture_call(context, target=str(demo))
assert "Cell id=" in file_text
assert "## Setup" in file_text
assert "def demo_fn(x):" in file_text
assert "class DemoBox:" in file_text
assert "1 |" not in file_text

out = StringIO()
with redirect_stdout(out):
    file_result = context(str(demo))
assert "## Setup" in out.getvalue()
assert "## Setup" in file_result["text"]

chapter = capture_call(context, target="Setup", scope=str(demo))
assert "## Setup" in chapter
assert "value = 3" in chapter

doc = capture_call(context, target="context", scope=str(project_root / "nbs/01_read.ipynb"), overview=True)
assert "Implementation context: context" in doc
assert "Implementation" in doc

## Writing and targeted updates

The write path should append parsed Markdown and code blocks. The update path should use a stable cell id, then preserve the id while replacing the source.

In [ ]:
write_nb(
    str(demo),
    "%%markdown\n## Result\nThe next cell is edited by id.\n---\n%%code\nresult = value + 4\nresult",
)

result_cell = next(cell for cell in read_nb(demo).cells if "result = value + 4" in cell.source)
old_id = result_cell.id

update_cell(
    str(demo),
    "result = value + 5\nresult",
    cell_id=old_id,
)

updated = next(cell for cell in read_nb(demo).cells if cell.id == old_id)
assert "value + 5" in updated.source

## Execution and review output

Executing the fixture should store outputs in the notebook. Reviewing a repository notebook against itself should report no code-cell changes, which keeps review noise low.

In [ ]:
exec_nb(str(demo), timeout=5, show_output=False, allow_new=True)

executed = next(cell for cell in read_nb(demo).cells if cell.id == old_id)
texts = []
for output in executed.outputs:
    data = output.get("data", {})
    if "text/plain" in data:
        text = data["text/plain"]
        texts.append("".join(text) if isinstance(text, list) else str(text))
assert any("8" in text for text in texts)

diff_text = capture_call(diff_nb, path=str(project_root / "nbs/02_write.ipynb"), ref_a=None)
assert "No code cell changes" in diff_text

In [ ]:
safe_helper_root = demo_path("test_nbskill_safe_helper", base="/private/tmp")
safe_helper_root.mkdir()
try:
    safe_helper_nb = safe_helper_root / "safe_helper.ipynb"
    _write_nb(new_nb([
        mk_cell(chr(10).join([
            "def renamed_helper(x):",
            "    return x + 1",
            "",
            "def call_old(x):",
            "    return renamed_helper(x)",
            "",
            "assert call_old(2) == 3",
        ]), cell_type="code"),
    ]), safe_helper_nb)
    safe_helper_text = capture_call(exec_nb, path=str(safe_helper_nb), safe=True, allow_new=True, check_only=True, show_output=True, timeout=5)
    assert "NameError" not in safe_helper_text

    out = StringIO()
    with redirect_stdout(out):
        diff_nb(str(safe_helper_nb), ref_a=None, ref_b=None)
    disposable_diff = out.getvalue()
    assert "--- code cell" in disposable_diff and "+def renamed_helper" in disposable_diff
finally:
    remove_demo_path(safe_helper_root)

## Conversion and MCP construction

The converter should create a valid nbdev notebook from Python source, and the MCP factory should be testable without starting a server.

In [ ]:
sample_py = root / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted = root / "sample_tool.ipynb"

convert(str(sample_py), dest=str(converted))
converted_nb = read_nb(converted)
assert converted.exists()
assert "#| default_exp sample_tool" in converted_nb.cells[0].source
assert any("def double" in cell.source for cell in converted_nb.cells)

mcp = create_mcp()
assert mcp is not None

remove_demo_path(root)

In [ ]:
def _write_split_contract_notebook(path):
    _write_nb(new_nb([
        mk_cell("#| default_exp split_source", cell_type="code"),
        mk_cell("import math", cell_type="code"),
        mk_cell(chr(10).join(["#| export", "def _helper(x):", "    return math.ceil(x)"]), cell_type="code"),
        mk_cell("## Feature", cell_type="markdown"),
        mk_cell(chr(10).join(["#| export", "def split_value(x):", "    return _helper(math.sqrt(x))"]), cell_type="code"),
        mk_cell("## Keep", cell_type="markdown"),
        mk_cell(chr(10).join(["#| export", "def keep_value(x):", "    return _helper(x)"]), cell_type="code"),
    ]), path)


In [ ]:
split_root = demo_path("test_nbskill_split")
try:
    split_root.mkdir()
    split_src = split_root / "split_source.ipynb"
    split_dest = split_root / "split_dest.ipynb"
    _write_split_contract_notebook(split_src)
    plan_text = capture_call(
        split_nb_chapter,
        path=str(split_src), chapter="Feature", dest=str(split_dest),
        default_exp="split_dest", dry_run=True,
    )
    assert "would split" in plan_text
    assert "_helper -> helper" in plan_text
    assert not split_dest.exists()
    split_nb_chapter(str(split_src), "Feature", str(split_dest), default_exp="split_dest", dry_run=False)
    src_text = "\n".join(cell.source for cell in read_nb(split_src).cells)
    dest_text = "\n".join(cell.source for cell in read_nb(split_dest).cells)
    assert "## Feature" not in src_text
    assert "## Keep" in src_text
    assert "def helper" in src_text
    assert "return helper(x)" in src_text
    assert "#| default_exp split_dest" in dest_text
    assert "import math" in dest_text
    assert "from nbskill.split_source import helper as _helper" in dest_text
    assert "def split_value" in dest_text
    assert "_helper(math.sqrt(x))" in dest_text
finally:
    remove_demo_path(split_root)


## MCP concurrency and wrapper failures

MCP tools should preserve notebook locking when clients issue parallel calls, and a failing wrapper should not leave the server unable to handle a later call for the same notebook.

In [ ]:
import asyncio
import threading
import time
import nbskill.mcp as _mcp_mod


In [ ]:
async def _exercise_mcp_context_read(parallel_root):
    parallel_root.mkdir()
    path = parallel_root / "sample.ipynb"
    _write_nb(new_nb([mk_cell("## Sample", cell_type="markdown")]), path)
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool("context", {"target": str(path)})
    assert result.structured_content["context"]["resolved_kind"] == "notebook"
    assert "Sample" in result.structured_content["full_output"]
    debug_path = project_root / "nbs/data/test_nbskill.ipynb"
    debug_result = await mcp.call_tool("context", {"target": str(debug_path), "overview": False, "detail": "debug"})
    assert debug_result.structured_content["context"]["resolved_kind"] == "notebook"


async def _exercise_mcp_parallel_review(parallel_root):
    parallel_root.mkdir()
    path = parallel_root / "review.ipynb"
    _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), path)
    mcp = _mcp_mod.create_mcp()
    results = await asyncio.gather(
        mcp.call_tool("style_check", {"path": str(path), "max_output_chars": 2000}),
        mcp.call_tool("doctor", {"path": str(path), "scopes": "error"}),
        mcp.call_tool("diff_nb", {"path": str(path)}),
    )
    assert all(hasattr(result, "structured_content") for result in results)
    assert results[2].structured_content["ok"] is False
    assert "disposable notebook" in results[2].structured_content["full_output"]
    health = await mcp.call_tool("healthcheck", {})
    assert "nbskill mcp ok" in health.structured_content["full_output"]

In [ ]:
parallel_root = demo_path("test_mcp_context_read")
try:
    await _exercise_mcp_context_read(parallel_root)
finally:
    remove_demo_path(parallel_root)

parallel_review_root = demo_path("test_mcp_parallel_review")
try:
    await _exercise_mcp_parallel_review(parallel_review_root)
finally:
    remove_demo_path(parallel_review_root)

In [ ]:
import nbskill.mcp as _mcp_mod

failure_root = demo_path("test_mcp_wrapper_failure")
try:
    failure_root.mkdir()
    failure_path = failure_root / "failure.ipynb"
    _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), failure_path)

    old_context = _mcp_mod.context

    def failing_context(*args, **kwargs):
        raise RuntimeError("intentional wrapper failure")

    try:
        _mcp_mod.context = failing_context
        mcp = _mcp_mod.create_mcp()
        failed = await mcp.call_tool("context", {"target": str(failure_path)})
        assert failed.structured_content["ok"] is False
        assert "intentional wrapper failure" in failed.structured_content["full_output"]
    finally:
        _mcp_mod.context = old_context

    recovered = await mcp.call_tool("context", {"target": str(failure_path)})
    assert recovered.structured_content["context"]["resolved_kind"] == "notebook"
finally:
    remove_demo_path(failure_root)